## MQA, GQA, LMA

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
class MQA(nn.Module):
  def __init__(self, d_model, num_heads):
    super(MQA, self).__init__()
    self.num_heads = num_heads
    self.d_k = d_model // num_heads

    self.W_Q = nn.Linear(d_model, d_model) # d_model is same as d_k * h
    self.W_K = nn.Linear(d_model, self.d_k)
    self.W_V = nn.Linear(d_model, self.d_k)
    self.W_O = nn.Linear(d_model, d_model)



  def forward(self, x):
    '''
    Input:
      x(input): (N, T, d_model)

    Output:
      self.W_O(out): (N, T, d_model)
    '''
    N, T, _ = x.size()

    # Get Q for all heads and K, V just one for one single head
    Q = self.W_Q(x).view(N, T, self.num_heads, self.d_k).transpose(1, 2) # (N, h, T, d_k)
    K = self.W_K(x).unsqueeze(1) # (N, 1, T, d_k)
    V = self.W_V(x).unsqueeze(1) # (N, 1, T, d_k)

    # Calculate attention with Q, K, V
    scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5) #(N, h, T, T)
    attn = F.softmax(scores, dim=-1) #(N, h, T, T)
    out = torch.matmul(attn, V) #(N, h, T, d_k)
    out = out.transpose(1, 2).contiguous().view(N, T, -1)

    return self.W_O(out)

In [3]:
class GQA(nn.Module):
  def __init__(self, d_model, num_heads, num_kv_groups=8):
    super(GQA, self).__init__()

    self.num_heads = num_heads
    self.num_kv_groups = num_kv_groups
    self.heads_per_group = num_heads // num_kv_groups
    self.d_k = d_model // num_heads

    self.W_Q = nn.Linear(d_model, d_model) # d_model equals to self.d_k * h
    self.W_K = nn.Linear(d_model, self.d_k * num_kv_groups)
    self.W_V = nn.Linear(d_model, self.d_k * num_kv_groups)
    self.W_O = nn.Linear(d_model, d_model)


  def forward(self, x):
    '''
    Inputs:
      x: (input): (N, T, d_model)
    Outputs:
      self.W_O(out): (N, T, d_model)

    '''
    N, T, _ = x.size()


    # get Q, K, V
    Q = self.W_Q(x).view(N, T, self.num_heads, self.d_k).transpose(1, 2) # (N, h, T, d_k)
    K = self.W_K(x).view(N, T, self.num_kv_groups, self.d_k).transpose(1, 2) # (N, g, T, d_k)
    V = self.W_V(x).view(N, T, self.num_kv_groups, self.d_k).transpose(1, 2) # (N, g, T, d_k)

    # calculate attention
    '''
    iterating number of groups, get current subset of Q, subset of K, and subset of V
      current_Q: (N, h // g, T, d_k)
      current_K: (N, 1, T, d_k)
      current_V: (N, 1, T, d_k)


      repeat current_K and current_V


      calcualte attention with current_Q, repeated_current_K, repeated_current_V

    But for efficient implementatin, just do the one time attention score calculation as follows:
    Q, repeated
    '''
    K = K.repeat_interleave(self.heads_per_group, dim=1)
    V = V.repeat_interleave(self.heads_per_group, dim=1)
    attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)
    attn_weights = F.softmax(attn_scores, dim=-1)
    out = torch.matmul(attn_weights, V)

    out = out.transpose(1, 2).contiguous().view(N, T, -1)

    return self.W_O(out)

In [4]:
class MLA(nn.Module):
  def __init__(self, d_model, num_heads, d_compressed):
    super(MLA, self).__init__()
    self.num_heads = num_heads
    self.d_k = d_model // num_heads
    self.d_c = d_compressed

    self.W_Q = nn.Linear(d_model, d_model)
    self.W_K = nn.Linear(d_model, d_model)
    self.W_V = nn.Linear(d_model, d_model)

    self.W_CQ = nn.Linear(self.d_k, self.d_c, bias=False)
    self.W_CK = nn.Linear(self.d_k, self.d_c, bias=False)
    self.W_CV = nn.Linear(self.d_k, self.d_c, bias=False)

    self.W_O = nn.Linear(num_heads * self.d_c, d_model)

  def forward(self, x):
    '''
    Inputs:
      x(input): (N, T, d_model)
    Outputs:
      self.W_O(out): (N, T, d_model)
    '''
    N, T, _ = x.size()

    # compute Q, K, V for all heads
    Q = self.W_Q(x).view(N, T, self.num_heads, self.d_k).transpose(1, 2)  # (N, h, T, d_k)
    K = self.W_K(x).view(N, T, self.num_heads, self.d_k).transpose(1, 2)  # (N, h, T, d_k)
    V = self.W_V(x).view(N, T, self.num_heads, self.d_k).transpose(1, 2)  # (N, h, T, d_k)

    # compress Q, K, V
    Q_c = self.W_CQ(Q) # (N, h, T, d_c)
    K_c = self.W_CK(K) # (N, h, T, d_c)
    V_c = self.W_CV(V) # (N, h, T, d_c)

    # calculate the attention
    scores = torch.matmul(Q_c, K_c.transpose(-2, -1)) / (self.d_c ** 0.5) # (N, h, T, T)
    attn = F.softmax(scores, dim=-1) # (N, h, T, T)
    out = torch.matmul(attn, V_c) # (N, h, T, d_c)
    out = out.transpose(1, 2).contiguous().view(N, T, -1)  # (N, T, h * d_c)
    return self.W_O(out)

## Test

In [5]:
batch_size = 2
seq_len = 10
d_model = 64
num_heads = 32
num_kv_groups = 2
d_compressed = 16

In [10]:
# create a random input
x = torch.randn(batch_size, seq_len, d_model)

# test MQA
mqa = MQA(d_model=d_model, num_heads=num_heads)
out_mqa = mqa(x)
print("\nMQA:")
print(f"input shape: {x.shape}")
print(f"output shape: {out_mqa.shape}")

# test GQA
gqa = GQA(d_model=d_model, num_heads=num_heads, num_kv_groups=num_kv_groups)
out_gqa = gqa(x)
print("\nGQA:")
print(f"input shape: {x.shape}")
print(f"output shape: {out_gqa.shape}")

# test MLA
mla = MLA(d_model=d_model, num_heads=num_heads, d_compressed=d_compressed)
out_mla = mla(x)
print("\nMLA:")
print(f"input shape: {x.shape}")
print(f"output shape: {out_mla.shape}")


MQA:
input shape: torch.Size([2, 10, 64])
output shape: torch.Size([2, 10, 64])

GQA:
input shape: torch.Size([2, 10, 64])
output shape: torch.Size([2, 10, 64])

MLA:
input shape: torch.Size([2, 10, 64])
output shape: torch.Size([2, 10, 64])
